# BM25 Ranking System for Legal Document Retrieval
This notebook demonstrates:
1. Verb extraction from questions
2. Concept and relation matching
3. Section retrieval via triplets
4. BM25 ranking of sections

In [1]:
import os
import sys
sys.path.append(r"E:\Github\LawAssistant")

from triplet_extraction.src.db import init_mongo
from triplet_extraction.src.triplet_extraction import init_vncorenlp
from retrieval.src.retrieval_system import retrieve_and_rank, display_results
import phonlp

## 1. Initialize Database and NLP Models

In [2]:
# Initialize MongoDB
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]
sections_col = db["legal_sections"]
concepts_col = db["concepts"]
relations_col = db["relations"]
triplets_col = db["triplets_new"]

You successfully connected to MongoDB!


In [3]:
# Initialize NLP models
vncorenlp_path = r"E:\Github\LawAssistant\triplet_extraction\nlp_models\VnCoreNLP-1.2"
phonlp_path = r"E:\Github\LawAssistant\triplet_extraction\nlp_models\phonlp"

vncorenlp_client = init_vncorenlp(vncorenlp_path)
phoNLP_model = phonlp.load(save_dir=phonlp_path)

print("NLP models loaded successfully!")

Loading model from: E:\Github\LawAssistant\triplet_extraction\nlp_models\phonlp/phonlp.pt
NLP models loaded successfully!


## 2. Test with a Question

In [4]:
# Test question
question = "Điều kiện chuyển nhượng quyền sử dụng đất là gì?"
print(f"Question: {question}")

Question: Điều kiện chuyển nhượng quyền sử dụng đất là gì?


## 3. Retrieve and Rank Sections

In [5]:
# Retrieve and rank using hybrid approach (BM25 + Triplet scoring)
ranked_sections = retrieve_and_rank(
    question=question,
    vncorenlp_client=vncorenlp_client,
    phoNLP_model=phoNLP_model,
    sections_col=sections_col,
    concepts_col=concepts_col,
    relations_col=relations_col,
    triplets_col=triplets_col,
    top_k=10,
    use_hybrid=True,
    bm25_weight=0.6,
    triplet_weight=0.4
)

Segmented question: điều_kiện chuyển_nhượng quyền sử_dụng đất là gì


100%|██████████| 1/1 [00:00<00:00, 11.82it/s]


Extracted verbs: ['chuyển_nhượng', 'sử_dụng', 'là']



Matched 30 concepts and 0 relations

Matched Concepts:
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → đất
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → quyền sử dụng đất
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → quyền
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → quyền sử dụng
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → chuyển nhượng quyền sử dụng đất
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → điều kiện
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → c
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → sử dụng đất
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → giấy phép
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → sử dụng
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → d
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → đ
  Position 0: 'điều_kiện chuyển_nhượng quyền sử_dụng đất' → sử dụng đ
  Position 0: 'điều_kiện chuyển_nh

## 4. Display Results

In [6]:
# Display ranked results
display_results(ranked_sections, sections_col)


=== RANKED RESULTS ===

--- Rank 1 ---
Section ID: c00bb12001d8fb0c3d7d792a4d75e5072809447f708de21cfdc49807071ea359
Hybrid Score: 0.6935
  - BM25 Score: 11.9140 (normalized: 0.8642)
  - Triplet Score: 28 (normalized: 0.4375)
Full Path: 101/2024/NĐ-CP_chương iii_mục 3_điều 30_khoản 13_điểm b
Content Preview: Hợp đồng chuyển nhượng hoặc hợp đồng chuyển giao khác về quyền sử dụng đất, quyền sở hữu tài sản gắn liền với đất giữa người có quyền chuyển nhượng, bán tài sản thế chấp là quyền sử dụng đất, tài sản ...

--- Rank 2 ---
Section ID: 4b66ffea2a2685fbffa06ec3adbf6a0b6940ffe2cb03a930345ae9c3c3216029
Hybrid Score: 0.6250
  - BM25 Score: 13.7862 (normalized: 1.0000)
  - Triplet Score: 4 (normalized: 0.0625)
Full Path: 31/2024/QH15_chương iii_mục 4_điều 44_khoản 3_điểm a
Content Preview: Trong trường hợp chuyển nhượng quyền sử dụng đất thì bên chuyển nhượng trong hợp đồng chuyển nhượng quyền sử dụng đất là người nhận thừa kế;

--- Rank 3 ---
Section ID: 92f29b3a0c8788749049928ec7e0393eddb

## 5. Try BM25-only Ranking (Without Triplet Scores)

In [ ]:
# Retrieve and rank using BM25 only
ranked_sections_bm25 = retrieve_and_rank(
    question=question,
    vncorenlp_client=vncorenlp_client,
    phoNLP_model=phoNLP_model,
    sections_col=sections_col,
    concepts_col=concepts_col,
    relations_col=relations_col,
    triplets_col=triplets_col,
    top_k=10,
    use_hybrid=False  # BM25 only
)

display_results(ranked_sections_bm25, sections_col)

## 6. Compare Different Questions

In [ ]:
test_questions = [
    "Điều kiện để được cấp giấy chứng nhận quyền sử dụng đất là gì?",
    "Ai có quyền chuyển nhượng quyền sử dụng đất?",
    "Thủ tục thu hồi đất để thực hiện dự án đầu tư như thế nào?",
]

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*100}")
    print(f"QUESTION {i}: {q}")
    print('='*100)
    
    results = retrieve_and_rank(
        question=q,
        vncorenlp_client=vncorenlp_client,
        phoNLP_model=phoNLP_model,
        sections_col=sections_col,
        concepts_col=concepts_col,
        relations_col=relations_col,
        triplets_col=triplets_col,
        top_k=5,
        use_hybrid=True
    )
    
    display_results(results, sections_col)

## 7. View Detailed Concept and Relation Matching

In [ ]:
# Get detailed matching information
from retrieval.src.retrieval_system import print_matched_concepts_relations

ranked_sections, matched_concepts, matched_relations = retrieve_and_rank(
    question=question,
    vncorenlp_client=vncorenlp_client,
    phoNLP_model=phoNLP_model,
    sections_col=sections_col,
    concepts_col=concepts_col,
    relations_col=relations_col,
    triplets_col=triplets_col,
    top_k=5,
    use_hybrid=True,
    return_matches=True  # Return matching details
)

# Print detailed matching information
print_matched_concepts_relations(matched_concepts, matched_relations)

## 8. Inspect Individual Components

In [ ]:
# Inspect verb extraction
from retrieval.src.retrieval_system import extract_verbs, match_concepts_and_relations
from triplet_extraction.src.triplet_extraction import clean_text

test_q = "Phải xác nhận tài sản trên đất mới được bán đất có đúng không?"
cleaned = clean_text(test_q)
segmented = vncorenlp_client.word_segment(cleaned)[0]

print("Original:", test_q)
print("Cleaned:", cleaned)
print("Segmented:", segmented)

verbs = extract_verbs(segmented, phoNLP_model)
print("\nExtracted Verbs:")
for v in verbs:
    print(f"  - {v['word']} (head: {v['head']}, deprel: {v['deprel']})")

In [ ]:
# Inspect concept/relation matching
segmented_tokens = segmented.split(" ")
all_concepts = list(concepts_col.find({}))
all_relations = list(relations_col.find({}))

matched_concepts, matched_relations = match_concepts_and_relations(
    segmented_tokens, all_concepts, all_relations
)

print("\nMatched Concepts:")
for m in matched_concepts:
    print(f"  Position {m['position']}: '{m['matched_text']}' -> {m['data']['name']}")

print("\nMatched Relations:")
for m in matched_relations:
    print(f"  Position {m['position']}: '{m['matched_text']}' -> {m['data']['name']}")